## Gradient data with Peltier on one side and RT on other

note: it was difficult to keep the RT side consistent, so analysis tries to take temperature gaps into account as well as whether the battery was heating or cooling

In [1]:
import sys
sys.path.append('..') # path to the src directory
sys.path.append('/Users/xz498/Desktop/ultrasound project/data analysis/ultrasonicTesting')

import pickleJar as pj
import sqliteUtils as squ
import numpy as np
from numpy import fft
import os.path
import matplotlib.pyplot as plt

In [2]:
# First we load the data from the file we want to analyze
# The experiment should always output sqlite3 files, so let's convert them to a more usable pickle form first
sqliteFile = "/Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/test_holder_xz_battery_sink1776806811.sqlite3"

# Convert the sqlite3 to a .pickle
# This takes a few seconds. A progress bar will display in your terminal
pj.sqliteToPickle(sqliteFile)

# Load the pickle
pickleFile = os.path.splitext(sqliteFile)[0] + '.pickle'

data = pj.loadPickle("/Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/test_holder_xz_battery_sink1776806811.pickle")

sqliteToPickle Warning: pickle file /Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/test_holder_xz_battery_sink1776806811.pickle already exists. Conversion aborted.


## Merge the waveform and controller data wrt time that UT is running

In [3]:
def correct_voltage(dataframe):
    '''correct measurements by matching mean of 1st 25 pts to 0.
    '''
    for item in dataframe['voltage']:
        item -= item[:25].mean()
    return dataframe

In [4]:
import pandas as pd
measurements = {k:v for k,v in data.items() if isinstance(k, int)}
waveform_df = pd.DataFrame(measurements).transpose()
waveform_df = correct_voltage(waveform_df)

In [5]:
waveform_df

,voltage,time,time_collected,collection_index
0,"[0.028479822834645646, 0.05960679133858268, 0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776806812.911217,0.0
1,"[-0.22195127952755905, -0.1528075787401575, 0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776806813.964868,1.0
2,"[0.04356053149606298, -0.1292987204724409, 0.0...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776806814.4453,2.0
3,"[0.131373031496063, 0.2005167322834645, 0.0276...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776806815.481088,3.0
4,"[-0.1109756397637795, -0.1714763779527559, -0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776806816.488133,4.0
...,...,...,...,...
1791,"[-0.01848179133858263, 0.007170275590551167, 0...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776808607.976693,1791.0
1792,"[0.033088090551181115, 0.15458169291338586, 0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776808608.959206,1792.0
1793,"[0.031349655511811014, -0.141263533464567, -0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776808609.974871,1793.0
1794,"[0.04110605314960636, -0.1404576771653543, 0.2...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1776808610.961416,1794.0


In [7]:
from datetime import datetime
controller_monitor_df = pd.read_csv('/Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/CoSo_2026-04-21_18-07-51_Monitor.csv',
                            delimiter=';',
                            usecols=[   'Time', 'Milliseconds',
                                        '1000.1: CH1 Object', '1000.2: CH2 Object', 
                                        '1045.1: HR1 Temp', '1045.2: HR2 Temp',
                                    ])
controller_monitor_df['Time_str'] = controller_monitor_df['Time']
controller_monitor_df['Time'] = controller_monitor_df['Time'].apply(lambda x: datetime.strptime(x, '%m/%d/%Y %I:%M:%S %p').timestamp())
controller_monitor_df['Time'] += controller_monitor_df['Milliseconds']/1000
controller_monitor_df.drop(columns=['Milliseconds'], inplace=True)

ValueError: Usecols do not match columns, columns expected but not found: ['1000.1: CH1 Object', '1045.2: HR2 Temp', '1045.1: HR1 Temp', 'Time', 'Milliseconds', '1000.2: CH2 Object']

In [ ]:
controller_settings_df = pd.read_csv('/Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/CoSo_2026-04-21_18-07-51_Settings.csv',
                            delimiter=';',
                            usecols=[   'Time', 'Milliseconds',
                                        '2010.1: Output Enable', '2010.2: Output Enable', 
                                        '3034.1: Peltier Polarity', '3034.2: Peltier Polarity',
                                        '3051.1: CH1 Ramp', '3051.2: CH2 Ramp',
                                        '1044.1: LR1 Temp','1044.2: LR2 Temp'
                                    ])
# to correct later: PID params, max temperature change, )
controller_settings_df['Time'] = controller_settings_df['Time'].apply(lambda x: datetime.strptime(x, '%m/%d/%Y %I:%M:%S %p').timestamp())
controller_settings_df['Time'] += controller_settings_df['Milliseconds']/1000
controller_settings_df.drop(columns=['Milliseconds'], inplace=True)

In [ ]:
controller_df = pd.merge_asof(
    left=controller_monitor_df,
    right=controller_settings_df,
    left_on='Time',
    right_on='Time',
    direction='nearest' 
)

In [ ]:
waveform_df['time_collected'] = waveform_df['time_collected'].astype(float)
controller_monitor_df['Time'] = controller_monitor_df['Time'].astype(float)
controller_settings_df['Time'] = controller_settings_df['Time'].astype(float)

In [ ]:
merged_df = pd.merge_asof(
    left=waveform_df,
    right=controller_df,
    left_on='time_collected',
    right_on='Time',
    direction='nearest' 
)

In [ ]:
merged_df.to_csv('/Users/xz498/Desktop/ultrasound project/data analysis/gradient_data/merged_df.csv')

In [ ]:
merged_df.tail()

,voltage,time,time_collected,collection_index,Time,1000.1: CH1 Object,1000.2: CH2 Object,1045.1: HR1 Temp,1045.2: HR2 Temp,Time_str,1044.1: LR1 Temp,1044.2: LR2 Temp,2010.1: Output Enable,2010.2: Output Enable,3034.1: Peltier Polarity,3034.2: Peltier Polarity,3051.1: CH1 Ramp,3051.2: CH2 Ramp
1791,"[-0.01848179133858263, 0.007170275590551167, 0...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1.776809e+09,1791.0,1.776809e+09,21.467218,19.607660,21.467218,19.607660,4/21/2026 5:56:48 PM,22.973139,27.122614,ON,ON,Heating,Cooling,50.1,50.1
1792,"[0.033088090551181115, 0.15458169291338586, 0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1.776809e+09,1792.0,1.776809e+09,21.462091,19.546991,21.462091,19.546991,4/21/2026 5:56:49 PM,24.391596,35.920708,ON,ON,Heating,Cooling,50.1,50.1
1793,"[0.031349655511811014, -0.141263533464567, -0....","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1.776809e+09,1793.0,1.776809e+09,21.465631,19.493738,21.465631,19.493738,4/21/2026 5:56:50 PM,24.391596,35.920708,ON,ON,Heating,Cooling,50.1,50.1
1794,"[0.04110605314960636, -0.1404576771653543, 0.2...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1.776809e+09,1794.0,1.776809e+09,21.467676,19.437586,21.467676,19.437586,4/21/2026 5:56:51 PM,24.391596,35.920708,ON,ON,Heating,Cooling,50.1,50.1
1795,"[0.03871062992125984, 0.2721013779527559, -0.0...","[2000.0, 2002.0, 2004.0, 2006.0, 2008.0, 2010....",1.776809e+09,1795.0,1.776809e+09,21.466242,19.358301,21.466242,19.358301,4/21/2026 5:56:52 PM,24.391596,35.920708,ON,ON,Heating,Cooling,50.1,50.1


## Compare Curves of gradients vs means

In [ ]:
import plotly.graph_objects as go
from ipywidgets import interact
import ipywidgets as widgets
from datetime import datetime
import matplotlib.cm as cm


def plot_waveform(i):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=merged_df['time'][i], y=merged_df['voltage'][i], mode='lines'))
    fig.update_layout(
        title=f'Measurement {i} at ({ merged_df["Time_str"][i] })',
        xaxis_title='Time (ns)',
        yaxis_title='Voltage (mV)'
    )
    return fig

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(merged_df)-1,
    step=1,
    description='measurement:',
    continuous_update=False
)

interact(plot_waveform,i=slider)

interactive(children=(IntSlider(value=0, continuous_update=False, description='measurement:', max=1795), Outpu…

<function __main__.plot_waveform(i)>

In [ ]:
import plotly.graph_objects as go
from ipywidgets import interact
import ipywidgets as widgets
from datetime import datetime
import matplotlib.cm as cm


merged_df['means'] = merged_df[['1000.1: CH1 Object', '1000.2: CH2 Object']].mean(axis=1)
merged_df['diff'] = merged_df['1000.2: CH2 Object'] - merged_df['1000.1: CH1 Object']

fig = go.Figure()
fig.add_trace(go.Scatter(x=merged_df.index.astype(str) + ': ' + merged_df['Time_str'], y=merged_df['diff'], mode='lines', name='difference'))
fig.add_trace(go.Scatter(x=merged_df.index.astype(str) + ': ' + merged_df['Time_str'], y=merged_df['means'], mode='lines', name='mean'))
fig.update_layout(
    title=f'difference and mean of RT and Peltier sides',
    xaxis_title='Time',
    yaxis_title='Temperature (C)'
)
fig.show()

## filtered by mean temperature

In [ ]:
import matplotlib.cm as cm

def get_color(color_val):
    '''color_val is a value between 0 and 1'''
    viridis = cm.get_cmap('viridis')
    r, g, b, _ = viridis(color_val)
    return f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'

def plotly_contstrain_T(t1,t2, heating='both', sort_by='diff', index_range=None, direction='forward'):
    '''heating is on, off, or both
    sort_by is 'diff' or 'means'
    index_range is a tuple of (start, end). You can look at the above plot to decide range
    direction is 'forward' or 'reverse'. Forward is Ch2-Ch1, reverse is Ch1-Ch2
    '''
    if index_range is not None:
        ranged_df = merged_df.iloc[index_range[0]:index_range[1]:len(ranged_df)//20]
    else:
        ranged_df = merged_df
    
    if heating == 'both':
        ranged_df = merged_df[(t1<=merged_df['means']) & (merged_df['means']<t2)]
    elif heating == 'on':
        ranged_df = merged_df[(t1<=merged_df['means']) & (merged_df['means']<t2) & \
                                (merged_df['2010.2: Output Enable'] == 'ON') & \
                                (merged_df['2010.1: Output Enable'] == 'ON')]
    elif heating == 'off':
        ranged_df = merged_df[(t1<=merged_df['means']) & (merged_df['means']<t2) & \
            (merged_df['2010.2: Output Enable'] == 'OFF') & \
            (merged_df['3034.2: Peltier Polarity'] == 'Reverse')]
    
    ranged_df.sort_values(by=sort_by, inplace=True, ascending=True if direction == 'forward' else False)
    ranged_df.reset_index(drop=True, inplace=True)

    fig = go.Figure()
    for index, row in ranged_df.iterrows():
        color_val = index/len(ranged_df)
        fig.add_trace(  go.Scatter(x=row['time'], y=row['voltage'], mode='lines',
                                  line=dict(width=1, color=get_color(color_val)),
                                  name=f'index {index}: {row["means"]:.2f} + {row["diff"]/2:.2f} C', 
                                  ) )        
    fig.update_layout(
        title=f'Voltage vs Time at means between {t1} and {t2} C',
        xaxis_title='Time (ns)',
        yaxis_title='Voltage (mV)'
    )
    return fig



In [ ]:
plotly_contstrain_T(29,31, heating='on', sort_by='diff', index_range=None, direction='forward')

/var/folders/vg/g8h80y317zj5sfz8_f4l85wddxx210/T/ipykernel_21645/3818058582.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  viridis = cm.get_cmap('viridis')
